# DeepDDS: graph-attention prediction of synergistic drug combinations
## A GPU-oriented reproduction of Wang et al. (2022)

This notebook explains and reproduces the central ideas in **DeepDDS: deep graph neural network with attention mechanism to predict synergistic drug combinations**.

- [Paper DOI](https://doi.org/10.1093/bib/bbab390)
- [Authors' public repository](https://github.com/Sinwang404/DeepDDs)

The authors' public repository currently contains no implementation or data. This reproduction therefore implements the described model directly and uses a frozen public DrugComb/TDC table. Raw/processed inputs and split metadata are stored under `data/DeepDDS/`; reusable weights are stored under `models/DeepDDS/`.

## 1. Task and biological context

For drugs $A$ and $B$ and cellular context $c$, predict whether their measured combined effect exceeds an additive expectation:

$$\hat y=f_\theta(G_A,G_B,c),\qquad y=\mathbb 1[s_{\mathrm{Loewe}}>10].$$

Each drug is a molecular graph; atoms are nodes and bonds define message-passing edges. The cellular branch represents the fact that the same pair can behave differently across cell lines. The model is a prioritization tool—not evidence of clinical efficacy or safety.

## 2. DeepDDS architecture

1. A **shared graph attention network (GAT)** encodes both drug graphs. Shared weights ensure the two drug positions use the same chemical representation.
2. Attention coefficients decide which neighboring atoms contribute to an atom update; gated attention pooling then identifies atoms important to the whole-drug representation.
3. A cell-line branch encodes gene expression when `data/DeepDDS/cell_expression.csv` is available. Otherwise, a learned cell-line embedding provides a runnable but less biologically informative fallback.
4. Symmetric pair features $h_A+h_B$, $|h_A-h_B|$, and $h_A\odot h_B$ are combined with the cell vector and classified. Symmetry prevents predictions from changing when drug order is swapped.

The symmetric fusion and modern PyG implementation are reproduction choices. The paper compares graph convolution and graph attention variants and uses cell-line molecular features.

In [ ]:
import hashlib
import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch_geometric.data import Batch, Data
from torch_geometric.nn import GATConv
from torch_geometric.utils import scatter, softmax

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
if USE_AMP:
    torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'DeepDDS' else Path.cwd()
DATA_DIR = PROJECT_ROOT/'data'/'DeepDDS'; MODEL_DIR = PROJECT_ROOT/'models'/'DeepDDS'
DATA_DIR.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'PyTorch={torch.__version__} device={device}')
if USE_AMP:
    p=torch.cuda.get_device_properties(0); print(f'{p.name}, {p.total_memory/2**30:.1f} GiB')

## 3. DrugComb data and leakage-aware choices

The bundled frozen Parquet file contains canonical columns `drug1_smiles`, `drug2_smiles`, `cell_line`, `synergy_score`, and `synergy_label`. We recreate the common balanced classification protocol by retaining all positives and a seeded equal-size sample of negatives. The balanced table and exact stratified split are persisted.

A row-random split can share drugs, pairs, and cell lines across partitions; it measures interpolation and is closest to many published protocols. For prospective discovery, use pair-held-out or drug-held-out splits. This notebook reports the limitation rather than presenting random-split performance as novel-pair generalization.

In [ ]:
RAW_PATH = DATA_DIR/'drugcomb_tdc_v1.1.15.parquet'
BALANCED_PATH = DATA_DIR/'balanced_pairs.parquet'
SPLIT_PATH = DATA_DIR/'split_seed42.pt'
if not RAW_PATH.exists():
    raise FileNotFoundError(f'Expected frozen DrugComb input at {RAW_PATH}')
if BALANCED_PATH.exists():
    pairs = pd.read_parquet(BALANCED_PATH)
else:
    raw = pd.read_parquet(RAW_PATH).dropna(subset=['drug1_smiles','drug2_smiles','synergy_score'])
    raw['cell_line'] = raw['cell_line'].fillna('UNKNOWN').astype(str)
    raw['synergy_label'] = (raw['synergy_score'].astype(float) > 10.0).astype(int)
    positive = raw[raw.synergy_label == 1]
    negative = raw[raw.synergy_label == 0].sample(n=len(positive), random_state=SEED)
    pairs = pd.concat([positive, negative]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    pairs.to_parquet(BALANCED_PATH, index=False)

def stratified_split(labels):
    generator = torch.Generator().manual_seed(SEED); result=[[],[],[]]
    for label in (0,1):
        ids=torch.tensor([i for i,y in enumerate(labels) if y==label])
        ids=ids[torch.randperm(len(ids), generator=generator)]
        a,b=int(.70*len(ids)),int(.85*len(ids))
        for bucket,chunk in zip(result,(ids[:a],ids[a:b],ids[b:])): bucket.extend(chunk.tolist())
    return {name: torch.tensor(sorted(ids)) for name,ids in zip(('train','val','test'),result)}
splits = torch.load(SPLIT_PATH, weights_only=True) if SPLIT_PATH.exists() else stratified_split(pairs.synergy_label.tolist())
if not SPLIT_PATH.exists(): torch.save(splits, SPLIT_PATH)
DATA_FINGERPRINT = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print(f'Balanced rows={len(pairs):,}; positive={pairs.synergy_label.mean():.1%}; cells={pairs.cell_line.nunique()}')
print({k:len(v) for k,v in splits.items()})

## 4. Convert SMILES to reusable molecular graphs

Atom features encode element, degree, formal charge, aromaticity, hybridization, and hydrogen count. Graphs are built once with RDKit and cached in `data/DeepDDS/drug_graphs.pt`; subsequent runs avoid repeated SMILES parsing.

In [ ]:
GRAPH_CACHE = DATA_DIR/'drug_graphs.pt'
ELEMENTS = [5,6,7,8,9,15,16,17,35,53]
HYBRIDIZATIONS = [Chem.HybridizationType.SP, Chem.HybridizationType.SP2,
                  Chem.HybridizationType.SP3, Chem.HybridizationType.SP3D]
def one_hot(value, choices): return [float(value==x) for x in choices]+[float(value not in choices)]
def atom_features(atom):
    return one_hot(atom.GetAtomicNum(), ELEMENTS)+one_hot(atom.GetDegree(), list(range(6)))+[
        atom.GetFormalCharge()/3, float(atom.GetIsAromatic())]+one_hot(atom.GetHybridization(), HYBRIDIZATIONS)+one_hot(atom.GetTotalNumHs(), list(range(5)))
def smiles_graph(smiles):
    mol=Chem.MolFromSmiles(smiles)
    if mol is None: raise ValueError(f'Invalid SMILES: {smiles}')
    x=torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float32)
    edges=[]
    for bond in mol.GetBonds():
        i,j=bond.GetBeginAtomIdx(),bond.GetEndAtomIdx(); edges.extend([(i,j),(j,i)])
    edge_index=torch.tensor(edges,dtype=torch.long).t().contiguous() if edges else torch.empty((2,0),dtype=torch.long)
    return (x,edge_index)
if GRAPH_CACHE.exists():
    graph_cache=torch.load(GRAPH_CACHE, weights_only=False)
else:
    unique=sorted(set(pairs.drug1_smiles)|set(pairs.drug2_smiles))
    graph_cache={s:smiles_graph(s) for s in unique}; torch.save(graph_cache, GRAPH_CACHE)
NODE_DIM=next(iter(graph_cache.values()))[0].shape[1]
print(f'Cached drugs={len(graph_cache):,}; atom features={NODE_DIM}; path={GRAPH_CACHE}')

## 5. Cellular context

If `cell_expression.csv` exists, its first column must be `cell_line` and remaining columns numeric gene features. The processed standardized tensor is cached. Without it, the model learns one embedding per cell-line identifier. That fallback captures repeated cell identity but cannot generalize mechanistically to unseen cell lines and is not equivalent to CCLE expression.

In [ ]:
EXPRESSION_PATH=DATA_DIR/'cell_expression.csv'; EXPRESSION_CACHE=DATA_DIR/'cell_expression_tensor.pt'
cell_names=sorted(pairs.cell_line.astype(str).unique()); cell_to_id={name:i for i,name in enumerate(cell_names)}
if EXPRESSION_PATH.exists():
    expression_df=pd.read_csv(EXPRESSION_PATH).set_index('cell_line').reindex(cell_names)
    if expression_df.isna().all(axis=1).any(): raise ValueError('Expression file misses one or more DrugComb cell lines.')
    expression_df=expression_df.apply(pd.to_numeric,errors='coerce').fillna(expression_df.median())
    values=torch.tensor(expression_df.values,dtype=torch.float32)
    values=(values-values.mean(0))/(values.std(0).clamp_min(1e-6))
    torch.save({'cell_names':cell_names,'values':values},EXPRESSION_CACHE)
    CELL_MODE='expression'
else:
    values=None; CELL_MODE='embedding'
print(f'Cell mode={CELL_MODE}; cell lines={len(cell_names)}; input dimension={None if values is None else values.shape[1]}')

In [ ]:
class PairDataset(Dataset):
    def __init__(self, indices): self.indices=indices.tolist()
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        row=pairs.iloc[self.indices[i]]
        return row.drug1_smiles,row.drug2_smiles,cell_to_id[str(row.cell_line)],float(row.synergy_label),self.indices[i]
def collate(rows):
    s1,s2,cells,labels,ids=zip(*rows)
    def batch_graph(smiles):
        return Batch.from_data_list([Data(x=graph_cache[s][0],edge_index=graph_cache[s][1]) for s in smiles])
    return batch_graph(s1),batch_graph(s2),torch.tensor(cells),torch.tensor(labels),torch.tensor(ids)
BATCH_SIZE=512 if USE_AMP else 64; NUM_WORKERS=min(8,os.cpu_count() or 1) if USE_AMP else 0
kwargs=dict(batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,pin_memory=USE_AMP,persistent_workers=NUM_WORKERS>0,collate_fn=collate)
loaders={k:DataLoader(PairDataset(v),shuffle=(k=='train'),**kwargs) for k,v in splits.items()}
print(f'batch={BATCH_SIZE}; workers={NUM_WORKERS}')

## 6. Shared GAT with atom attention

For an edge $i\leftarrow j$, GAT learns $\alpha_{ij}$ and updates $h_i'=\Vert_k\sigma(\sum_j\alpha_{ij}^{(k)}Wh_j)$. A second learned gate supplies normalized graph-level atom weights. Both drug orders pass through the same encoder.

In [ ]:
class DrugEncoder(nn.Module):
    def __init__(self,in_dim,hidden=128):
        super().__init__(); self.gat1=GATConv(in_dim,64,heads=4,dropout=.15)
        self.gat2=GATConv(256,hidden,heads=1,dropout=.15); self.norm=nn.LayerNorm(hidden)
        self.gate=nn.Sequential(nn.Linear(hidden,64),nn.Tanh(),nn.Linear(64,1))
    def forward(self,graph,return_attention=False):
        x=F.elu(self.gat1(graph.x,graph.edge_index)); x=self.norm(F.elu(self.gat2(x,graph.edge_index)))
        weights=softmax(self.gate(x).squeeze(-1),graph.batch)
        pooled=scatter(x*weights[:,None],graph.batch,dim=0,reduce='sum')
        return (pooled,weights) if return_attention else pooled
class CellEncoder(nn.Module):
    def __init__(self,n_cells,expression):
        super().__init__(); self.use_expression=expression is not None
        if self.use_expression:
            self.register_buffer('expression',expression); self.net=nn.Sequential(nn.Linear(expression.shape[1],256),nn.ReLU(),nn.Dropout(.2),nn.Linear(256,128))
        else: self.embedding=nn.Embedding(n_cells,128)
    def forward(self,ids): return self.net(self.expression[ids]) if self.use_expression else self.embedding(ids)
class DeepDDS(nn.Module):
    def __init__(self):
        super().__init__(); self.drug=DrugEncoder(NODE_DIM); self.cell=CellEncoder(len(cell_names),values)
        self.classifier=nn.Sequential(nn.Linear(512,256),nn.ReLU(),nn.Dropout(.3),nn.Linear(256,64),nn.ReLU(),nn.Linear(64,1))
    def forward(self,g1,g2,cell_ids,return_attention=False):
        if return_attention:
            d1,a1=self.drug(g1,True); d2,a2=self.drug(g2,True)
        else: d1,d2=self.drug(g1),self.drug(g2)
        c=self.cell(cell_ids); fused=torch.cat([d1+d2,(d1-d2).abs(),d1*d2,c],-1)
        logits=self.classifier(fused).squeeze(-1)
        return (logits,a1,a2) if return_attention else logits
model=DeepDDS().to(device); model.eval()
g1,g2,c,y,_=next(iter(loaders['train'])); g1,g2,c=g1.to(device),g2.to(device),c.to(device)
with torch.no_grad(): output=model(g1,g2,c)
print(f'Parameters={sum(p.numel() for p in model.parameters()):,}; output={output.shape}')
with torch.no_grad(): assert torch.allclose(model(g1,g2,c),model(g2,g1,c),atol=1e-5)
print('Drug-order symmetry check passed.')

In [ ]:
def binary_metrics(target,score):
    order=score.argsort(); ranks=torch.empty_like(order,dtype=torch.float); ranks[order]=torch.arange(1,len(score)+1,dtype=torch.float)
    pos=target==1; p,n=pos.sum().item(),(~pos).sum().item(); auroc=(ranks[pos].sum()-p*(p+1)/2)/max(1,p*n)
    desc=score.argsort(descending=True); sorted_y=target[desc]; precision_curve=sorted_y.cumsum(0)/torch.arange(1,len(target)+1)
    ap=(precision_curve*sorted_y).sum()/max(1,p); pred=score>=.5
    tp=(pred&pos).sum(); fp=(pred&~pos).sum(); fn=(~pred&pos).sum()
    precision=tp/(tp+fp).clamp_min(1); recall=tp/(tp+fn).clamp_min(1); f1=2*precision*recall/(precision+recall).clamp_min(1e-8)
    return {'auroc':float(auroc),'ap':float(ap),'accuracy':float((pred==target.bool()).float().mean()),'f1':float(f1)}
@torch.no_grad()
def evaluate(loader,return_values=False):
    model.eval(); scores=[];targets=[];ids=[]
    for g1,g2,c,y,row_ids in loader:
        g1,g2,c=g1.to(device,non_blocking=True),g2.to(device,non_blocking=True),c.to(device,non_blocking=True)
        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=USE_AMP): logits=model(g1,g2,c)
        scores.append(logits.sigmoid().cpu()); targets.append(y); ids.append(row_ids)
    scores,targets,ids=map(torch.cat,(scores,targets,ids)); result=binary_metrics(targets,scores)
    return (result,scores,targets,ids) if return_values else result

In [ ]:
EPOCHS,PATIENCE,FORCE_RETRAIN=100,15,False
CHECKPOINT=MODEL_DIR/'deepdds_gat.pt'
CONFIG={'node_dim':NODE_DIM,'cell_mode':CELL_MODE,'cells':cell_names,'seed':SEED,'data_fingerprint':DATA_FINGERPRINT}
history={'loss':[],'val_auc':[]}; best_auc=-1.; start=time.time()
if CHECKPOINT.exists() and not FORCE_RETRAIN:
    checkpoint=torch.load(CHECKPOINT,map_location=device,weights_only=True)
    if checkpoint['config']!=CONFIG: raise ValueError('Incompatible checkpoint; set FORCE_RETRAIN=True.')
    model.load_state_dict(checkpoint['model_state']); best_auc=checkpoint['best_auc']; print('Loaded',CHECKPOINT)
else:
    optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-5,fused=USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP); stale=0
    for epoch in range(1,EPOCHS+1):
        model.train(); total=count=0
        for g1,g2,c,y,_ in loaders['train']:
            g1,g2,c,y=g1.to(device,non_blocking=True),g2.to(device,non_blocking=True),c.to(device,non_blocking=True),y.to(device,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=USE_AMP): loss=F.binary_cross_entropy_with_logits(model(g1,g2,c),y)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),5.)
            scaler.step(optimizer); scaler.update(); total+=loss.item()*len(y); count+=len(y)
        val=evaluate(loaders['val']); history['loss'].append(total/count); history['val_auc'].append(val['auroc'])
        if val['auroc']>best_auc:
            best_auc,stale=val['auroc'],0; torch.save({'model_state':model.state_dict(),'config':CONFIG,'best_auc':best_auc},CHECKPOINT)
        else: stale+=1
        print(f"{epoch:03d} loss={total/count:.4f} val_auc={val['auroc']:.4f} val_ap={val['ap']:.4f}")
        if stale>=PATIENCE: print('Early stopping.'); break
    model.load_state_dict(torch.load(CHECKPOINT,map_location=device,weights_only=True)['model_state'])
test_metrics,test_score,test_target,test_ids=evaluate(loaders['test'],True)
print('Test:',json.dumps(test_metrics,indent=2),f'elapsed={(time.time()-start)/60:.1f} min')
predictions=pairs.iloc[test_ids.tolist()].copy(); predictions['predicted_probability']=test_score.numpy()
predictions.to_csv(DATA_DIR/'test_predictions.csv',index=False)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].plot(history['loss']); axes[0].set(title='Training loss',xlabel='epoch')
axes[1].plot(history['val_auc']); axes[1].set(title='Validation AUROC',xlabel='epoch')
plt.tight_layout(); plt.show()

## 7. Inspect atom attention

Attention weights reveal which atoms most influence a graph representation for this model. They are useful hypotheses, not causal attributions or interaction-site proof. The next cell highlights atoms for the first drug in one test batch.

In [ ]:
g1,g2,c,y,row_ids=next(iter(loaders['test'])); g1,g2,c=g1.to(device),g2.to(device),c.to(device)
model.eval()
with torch.no_grad(): logits,a1,a2=model(g1,g2,c,True)
row=pairs.iloc[int(row_ids[0])]; mol=Chem.MolFromSmiles(row.drug1_smiles)
weights=a1[g1.batch==0].float().cpu(); weights=(weights-weights.min())/(weights.max()-weights.min()+1e-8)
colors={i:(1.0,float(1-w),float(1-w)) for i,w in enumerate(weights)}
image=Draw.MolToImage(mol,size=(500,350),highlightAtoms=list(range(len(weights))),highlightAtomColors=colors)
print(f'Cell={row.cell_line}; true={int(row.synergy_label)}; predicted={logits[0].sigmoid().item():.3f}')
display(image)

## 8. Reproduction boundaries and stronger experiments

### Reproduced

- molecular graph inputs and shared graph-attention drug encoding;
- attention-weighted molecular representations;
- cell-context-aware binary synergy prediction at a Loewe threshold of 10;
- balanced DrugComb classification, AUROC/AP/F1 evaluation, and atom-attention inspection.

### Important differences

- The public author repository is empty, so this is a clean PyTorch/PyG implementation rather than execution of released author code.
- The frozen TDC DrugComb snapshot and random split need not match the paper's exact curation and folds.
- Learned cell embeddings are only a fallback. Supply CCLE-style expression features for a closer biological reproduction.
- Symmetric fusion enforces the unordered nature of a drug pair and differs from simple ordered concatenation.
- Random-row performance does not establish generalization to unseen drugs or pairs.

For discovery-quality evaluation, canonicalize unordered pairs before splitting, hold out complete pairs or drugs, retain the natural class imbalance with weighted loss, report PR-AUC and calibrated probabilities, compare Morgan-fingerprint baselines, bootstrap confidence intervals, and test multiple seeds.

## Reference

Wang J, Liu X, Shen S, et al. *DeepDDS: deep graph neural network with attention mechanism to predict synergistic drug combinations.* Briefings in Bioinformatics. 2022;23(1):bbab390.